# **Task 1: Filtering and Visualizing Imagery in Google Earth Engine**


In this first task, you will learn how to **access and filter satellite imagery in Google Earth Engine (GEE)**.  
This is the foundation for almost any remote sensing workflow, because raw collections are massive, and we must carefully select images that match our study needs.  

Specifically, you will:  
- Define an **Area of Interest (AOI)**.  
- Load Sentinel-2 imagery and **filter by location, date, and cloud cover**.  
- Select the **least cloudy image** for visualization.  
- Compute a simple **NDVI-based Forest/Non-Forest mask**.  
- **Export** results so you can use them outside GEE.  

By the end of this task, you should feel comfortable working with GEE image collections and generating your own first map products.  


## **Initial Setup**

We need to import the necessary Python package for this task. We also need to authenticate and initialize the Earth Engine package with our credentials.

If you don't have your credentials yet, please refer to the "***Registration to Earth Engine - Step by Step guide***" PDF on NTU Learn

In [1]:
import ee
import geemap


ee.Authenticate()
ee.Initialize(project="YOUR_PROJECTID") # Replace with your project's ID

***

#### A Note on `geemap` vs Google Earth Engine  

In this notebook we are using **geemap**, which is a Python package designed to make working with the **Google Earth Engine (GEE) Python API** easier.  

- **Google Earth Engine (GEE)**: the cloud platform where all the satellite imagery and computation happens. It has its own JavaScript Code Editor and a Python API.  
- **geemap**: an open-source Python library that builds on top of the GEE Python API. It provides:  
  - Interactive maps inside Jupyter/Colab (based on `ipyleaflet` or `folium`).  
  - Convenient helper functions for exporting, charting, and visualizing results.  
  - A smoother learning curve for those familiar with GIS or Python.  

Think of geemap as a **bridge**: it does not replace Earth Engine, but it makes it easier to use Earth Engine’s capabilities inside Python notebooks.  

| Feature | Google Earth Engine (GEE) Python API | `geemap` |
| :--- | :--- | :--- |
| **Core Function** | ⚙️ Provides the direct connection to GEE's data and servers. | ✨ A "wrapper" that makes the API easier to use. |
| **Interactive Map** | ❌ No built-in interactive map for notebooks. | ✅ Excellent interactive maps (`ipyleaflet` & `folium`). |
| **Learning Curve** | 🧗‍♂️ Steeper. Requires more manual setup for visualization. | 🚀 Much gentler. Designed for a smooth start. |
| **Analogy** | The car engine. | The car's dashboard and steering wheel. |
| **When to use it?**| It's the fundamental layer; `geemap` uses it underneath! | **Our default choice** for interactive analysis in notebooks. |

***


##**Step 1. Define Your Area of Interest (AOI)**

This is our first time using the **interactive map** provided by `geemap`. The map widget is similar to common web maps (Google Maps, OpenStreetMap):  
- You can **zoom** with the `+/-` buttons or your mouse wheel.  
- You can **pan** by clicking and dragging.  
- On the left, the toolbar lets you draw **points, lines, and polygons**.  


In this first step, we will define our **Area of Interest (AOI)** as a simple **point geometry**.   Later in the day, you will learn how to define polygons and use existing `FeatureCollections`.  

By displaying this AOI on the map, you can confirm the location you are working with.  


In [2]:
m = geemap.Map(center=[9.93, -84.09], zoom=8)

# Remove all existing basemap layers
m.layers = []

# Add Google Hybrid (satellite + labels) — no OSM policy issues
m.add_tile_layer(
    url="https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}",
    name="Google Hybrid",
    attribution="Google"
)

aoi = ee.Geometry.Point([-84.09, 9.93])
m.addLayer(aoi, {"color": "red"}, "AOI Point")
m

Map(center=[9.93, -84.09], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

## **Step 2. Load and Filter Sentinel-2 Imagery**  

Now that we have defined our Area of Interest (AOI), we can load satellite imagery from the **Sentinel-2 Surface Reflectance (SR) Harmonized collection**.  

This collection contains atmospherically corrected images from the Sentinel-2 mission, harmonized across sensors for consistency.  

Since Earth Engine stores **millions of Sentinel-2 images**, we need to filter them down to just the ones we care about.  

In this step, we will:  
1. **Filter by AOI** → only keep images covering our study area.  
2. **Filter by date range** → for example, the calendar year 2021.  
3. **Filter by cloud cover** → filter images to keep scenes with less than 10% cloud contamination.  

This way, we end up with a clean, manageable `ImageCollection` suitable for further analysis.  


In [3]:
s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
      .filterBounds(aoi) # 1) Keep only images that intersect our AOI
      .filterDate('2021-01-01', '2021-12-31')  # 2) Limit to the desired time range (calendar year 2021)
      .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10)))   # 3) Keep only scenes with <10% cloud cover
print("Total number of images in the filtered collection:", s2.size().getInfo())

Total number of images in the filtered collection: 5


## **Step 3. Select the Least Cloudy Image**  

Now that we have a filtered collection, we can select the **single least-cloudy image** from 2021.  
This is done by sorting the collection by the `CLOUDY_PIXEL_PERCENTAGE` metadata field and taking the first image in the list.  

Once we have that image, we need to **display it on the map**.  

_Remember:_ the map you created back in **Step 1** is still active, so after running this cell you will need to **scroll back to the top of the notebook** to see the new layer appear.  

To display imagery, we need two key pieces:  
1. A **visualization dictionary** (`vis`) that specifies which bands to show and how to stretch the pixel values.  
   - `bands=["B4", "B3", "B2"]` = the Red, Green, Blue channels → True Color.  
   - `min` and `max` = the range of pixel values stretched for display (Sentinel-2 reflectance values range roughly 0–3000).  
2. The **`add_layer()` function** from geemap, which adds a new layer to the interactive map.  
   - It takes the image, the `vis` parameters, and a label for the legend.  


In [4]:
# Select the least cloudy image by sorting by cloud percentage
least_cloudy = s2.sort("CLOUDY_PIXEL_PERCENTAGE").first()

# Define visualization parameters:
# - Use Sentinel-2 RGB bands (B4=Red, B3=Green, B2=Blue)
# - Stretch pixel values between 0 and 1500 for display
vis = {"bands":["B4","B3","B2"], "min":0, "max":1500}

# Add the least cloudy image to the existing map (from Step 1)
m.add_layer(least_cloudy, vis, "Least Cloudy 2021")

# Re-center the map around the selected image
m.centerObject(least_cloudy, 9)

print("[LAYER ADDED TO MAP]: The least cloudy image has been added to the map. You can scroll back up and visualize it")

[LAYER ADDED TO MAP]: The least cloudy image has been added to the map. You can scroll back up and visualize it


## **Step 4. Create a Forest/Non-Forest Mask**  

So far we have visualized the least-cloudy Sentinel-2 image. Now let’s take the next step: **deriving information** from the image.  

We will use the **Normalized Difference Vegetation Index (NDVI):**


$$
NDVI = \frac{NIR - RED}{NIR + RED}
$$



- **High NDVI values (close to 1)** → dense, healthy vegetation.  
- **Low or negative NDVI values** → water, bare soil, or urban areas.  

In this example, let's apply a **threshold of 0.8** to classify pixels:  
- NDVI > 0.8 → Forest (green)  
- NDVI ≤ 0.8 → Non-forest (red)  

This is a **simplified approach** (real-world forest mapping is more complex), but it demonstrates how we can move from raw imagery to a **binary classification mask**.


In [5]:
# Calculate NDVI from the least-cloudy image
# Sentinel-2: B8 = Near Infrared (NIR), B4 = Red
ndvi = least_cloudy.normalizedDifference(["B8", "B4"]).rename("NDVI")


# Creating a color palette suitable for visualizing NDVI
ndvi_palette = [
    '#440154', '#414487', '#2A788E', '#22A884',
    '#7AD151', '#FDE725'
]

# Add the NDVI layer to the map
m.add_layer(ndvi,{"min":-1, "max":1, 'palette': ndvi_palette},"NDVI")
print("[LAYER ADDED TO MAP]: NDVI has been added to the map")

# Apply a threshold to separate forest vs non-forest
# NDVI > 0.8 is classified as forest (True → 1), others as non-forest (False → 0)
forest = ndvi.gt(0.8)

# Add the mask to the map
# - Red = non-forest
# - Green = forest
#m.add_layer(forest, {"palette": ["red", "green"]}, "Forest Mask")
m.add_layer(forest, {"palette": ['#7B3294', '#4a914a']}, "Forest Mask")

print("[LAYER ADDED TO MAP]: Forest/Non-Forest Mask has been added to the map\n")

print("Take a few minutes to explore the NDVI and Forest Mask layers interactively.\nToggle layers, zoom in and out and observe how areas with dense vegetation\ncorrespond to higher NDVI values")
print("")

[LAYER ADDED TO MAP]: NDVI has been added to the map
[LAYER ADDED TO MAP]: Forest/Non-Forest Mask has been added to the map

Take a few minutes to explore the NDVI and Forest Mask layers interactively.
Toggle layers, zoom in and out and observe how areas with dense vegetation
correspond to higher NDVI values



## **Step 5. Export to Google Drive**

So far, everything we’ve done has stayed inside Google Earth Engine. That is powerful for exploration, but real-world projects (such as your final term project) often require bringing results into **other tools**:

- GIS software (e.g., QGIS, ArcGIS)  
- Data science frameworks (e.g., Python, R, MATLAB)  
- Reports, publications, or sharing with collaborators  

That is why this step shows you how to **export your results to your Google Drive**. We will export the **Forest/Non-Forest mask** as a `GeoTIFF` image.  

Key export parameters to notice:  
- **`folder`** → the name of the folder in Google Drive (if it doesn't exist, it will be created).  
- **`fileNamePrefix`** → the base name of the exported file.  
- **`scale`** → output resolution in meters. Sentinel-2 NDVI is ~10 m, but for quick tests in class you can use 10–20 m.  
- **`region`** → must be a polygon geometry (here, our AOI).  
- **`crs`** → coordinate reference system (EPSG:4326 = WGS84 lat/long).  
- **`maxPixels`** → increases the pixel limit for large exports.  

Once you run the export, check the **"Tasks" tab** in the Earth Engine Code Editor,  
or open the [Earth Engine Tasks Dashboard](https://console.cloud.google.com/earth-engine/tasks) to monitor progress.  

When the task completes, the file will appear in your **Google Drive** under the folder you specified.


In [6]:
# Use the geometry of the least cloudy image as export region
export_region = least_cloudy.geometry()

# --- Prepare the mask for export ---
forest_byte = forest.rename('forest').clip(export_region).toByte() # Replace .toByte() to .toFloat() if you are exporting a raster representing a continuous variable like NDWI

# --- Export to Google Drive as GeoTIFF ---
geemap.ee_export_image_to_drive(
    image=forest_byte,
    description='ForestMask_Export', # name of the task in EE Tasks
    folder='GEE_Task1_Exports', # Google Drive folder where the file will be saved
    fileNamePrefix='forest_mask', # File name of the output
    scale=100, # resolution in meters. The smaller the number, better the resolution.
    region=export_region,  # region or area that we want to export
    crs='EPSG:4326', #  # coordinate reference system
    maxPixels=1e13   # increase pixel limit
)

print("Export started. Check the Earth Engine Tasks Dashboard UI (link can be found above) to monitor progress.")

Export started. Check the Earth Engine Tasks Dashboard UI (link can be found above) to monitor progress.


###  Questions for Reflection  

- **Filtering**  
  - Why is filtering necessary when working with large satellite archives like Sentinel-2?  
  - How does changing the **date range** or **cloud threshold** affect the number of available images?  


- **Visualization**  
  - What happens if you change the visualization bands to `[B8, B4, B3]` (NIR, Red, Green)?  

- **NDVI & Forest Mask**  
  - What does NDVI tell us about vegetation health?  
  - Do you think NDVI > 0.8 is a good threshold for “forest”? Why or why not?  
  - Which land cover types might be misclassified by this simple threshold?  

- **Exporting**  
  - Why is exporting an essential step in an Earth Engine workflow?  
  - What happens if you export at a coarser resolution (e.g., 30 m instead of 10 m)?  
  - In a real-world project, would you prefer to export the **binary forest mask** or the **continuous NDVI image**? Why?  


# Optional Questions

### 1. Least-Cloudy NDWI (SAME AOI - Costa Rica)

- Compute the **Normalized Difference Water Index (NDWI)**
- Visualize NDWI (e.g., palette from **low → high water**) and **export it as a GeoTIFF**.


---

### 2. NDVI for Another Year & Site

- Choose a **different site** <u>(anywhere outside Costa Rica)</u> and a **different year**.  
- Visualize NDVI and **export it as a GeoTIFF**.


---

### 3. Reflection Questions
- Review and answer the guiding questions provided in the notebook.



# Bonus -  Batch Export Sentinel-2 Images to Google Drive

In [7]:
# -------------------------------------------------------------------
# BONUS: Batch Export Sentinel-2 Images to Google Drive
# -------------------------------------------------------------------
# This optional task shows how to export multiple images (e.g., all scenes
# in your filtered collection) as individual GeoTIFF files to Google Drive.
#
# Requirements:
#   - The variable `s2` must already be defined by running Step 2.
#   - Keep your AOI relatively small to avoid exceeding export limits.
#   - Limit the number of tasks to a reasonable number (e.g., 5–10) at a time.
# -------------------------------------------------------------------

# Define a small polygon AOI
geometry = ee.Geometry.Polygon(
    [[[-84.26925171677313, 10.041712558679492],
      [-84.26925171677313, 9.878048708943298],
      [-84.02205933396063, 9.878048708943298],
      [-84.02205933396063, 10.041712558679492]]],
    None, False
)

# Export settings
FOLDER_NAME = "GEE_Batch_Exports"
MAX_EXPORTS = 10     # limit number of exports (students can increase later)
SCALE = 10           # Sentinel-2 resolution (meters)
CRS = "EPSG:4326"

# Convert collection to list and limit number of images
s2_list = s2.toList(s2.size())
n = min(s2_list.size().getInfo(), MAX_EXPORTS)
print(f"Preparing {n} export tasks...")

# Loop through each image and export it
for i in range(n):
    img = ee.Image(s2_list.get(i)).select(['B4', 'B3', 'B2']).clip(geometry)
    date_str = ee.Date(img.get('system:time_start')).format('YYYYMMdd').getInfo()
    scene_id = img.get('system:index').getInfo()
    export_name = f"{scene_id}_{date_str}"

    task = ee.batch.Export.image.toDrive(
        image=img,
        description=export_name,
        folder=FOLDER_NAME,
        fileNamePrefix=export_name,
        region=geometry,
        scale=SCALE,
        crs=CRS,
        maxPixels=1e13
    )
    task.start()
    print(f"Started export task: {export_name}")

print("✅ All export tasks submitted.")
print("Check the Earth Engine Tasks Dashboard UI (link can be found above) to monitor progress.")


Preparing 5 export tasks...
Started export task: 20210224T160511_20210224T160840_T16PHR_20210224
Started export task: 20210224T160511_20210224T160840_T16PHS_20210224
Started export task: 20210306T160511_20210306T160616_T16PHR_20210306
Started export task: 20211211T160511_20211211T160510_T16PHR_20211211
Started export task: 20211211T160511_20211211T160510_T16PHS_20211211
✅ All export tasks submitted.
Check the Earth Engine Tasks Dashboard UI (link can be found above) to monitor progress.
